In [1]:
!pip install -q unsloth transformers datasets pandas tqdm groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 751.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
import json
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from groq import Groq

In [4]:
from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import whoami

login(token=userdata.get('HF_TOKEN_WRITE'))
print(whoami()['name'])

mfayazkhan


In [5]:
client = Groq(
    api_key=userdata.get('GROQ_API_KEY'),
)

In [6]:
SFT_MODEL = 'mfayazkhan/UrduGPT-v2'
DPO_MODEL = 'mfayazkhan/UrduGPT-v2-DPO'

In [8]:
model_sft, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=2048,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model_sft)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [9]:
evaluation_prompts = [
    "پاکستان کا دارالحکومت کیا ہے؟",
    "مشین لرننگ کیا ہے؟",
    "Python پروگرامنگ زبان کیوں استعمال کی جاتی ہے؟",
    "مصنوعی ذہانت کیا ہے؟",
    "SQL اور NoSQL میں کیا فرق ہے؟",
    "اسلام میں دیانت داری کی اہمیت کیا ہے؟",
    "ایک پروفیشنل ای میل کیسے لکھی جاتی ہے؟",
    "فری لانسنگ شروع کرنے کے لیے کیا کرنا چاہیے؟",
    "REST API کیا ہوتی ہے؟",
    "نیورل نیٹ ورک کیا ہوتا ہے؟"
]

In [10]:
print(f"Evaluation Samples: {len(evaluation_prompts)}")

Evaluation Samples: 10


In [16]:
def generate_response(model, tokenizer, prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    )

    return response.strip()

In [19]:
sft_results = []

for prompt in tqdm(evaluation_prompts):
  answer = generate_response(model_sft, tokenizer, prompt)

  sft_results.append({
      'prompt':'prompt',
      'sft_response': answer
  })

sft_df = pd.DataFrame(sft_results)
sft_df.head()

100%|██████████| 10/10 [02:41<00:00, 16.13s/it]


,prompt,sft_response
0,prompt,پاکستان کا دارالحکومت ایبادان ہے
1,prompt,مشین لرننگ ایک قسم کی کمپیوٹر سائنس ہے جو کمپی...
2,prompt,Python ایک بہترین پروگرامنگ زبان ہے کیونکہ یہ ...
3,prompt,مصنوعی ذہانت ایک قسم کی کمپیوٹر پروگرامنگ ہے ج...
4,prompt,SQL اور NoSQL دونوں ڈیٹا بیس سسٹم ہیں، لیکن ان...


In [27]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [28]:
model_dpo, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DPO_MODEL,
    max_seq_length=2048,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model_dpo)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load mfayazkhan/UrduGPT-v2-DPO as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Ll

In [30]:
dpo_results = []

for prompt in tqdm(evaluation_prompts):
  answer = generate_response(model_dpo, tokenizer, prompt)

  dpo_results.append({
      'prompt':'prompt',
      'dpo_response': answer
  })

dpo_df = pd.DataFrame(dpo_results)
dpo_df.head()

100%|██████████| 10/10 [01:32<00:00,  9.27s/it]


,prompt,dpo_response
0,prompt,پاکستان کا دارالحکومت اسلام آباد ہے۔
1,prompt,مشین لرننگ (Machine Learning) ایک فیلڈ ہے جو ک...
2,prompt,Python ایک بہت ہی مقبول اور موثر پروگرامنگ زبا...
3,prompt,مصنوعی ذہانت یا مصنوعی ذہانت (انگریزی: Artific...
4,prompt,SQL اور NoSQL دونوں ڈیٹا بزنس کے لیے ایک اہم ذ...


In [31]:
evaluation_df = sft_df.merge(dpo_df, on='prompt')
evaluation_df.head()

,prompt,sft_response,dpo_response
0,prompt,پاکستان کا دارالحکومت ایبادان ہے,پاکستان کا دارالحکومت اسلام آباد ہے۔
1,prompt,پاکستان کا دارالحکومت ایبادان ہے,مشین لرننگ (Machine Learning) ایک فیلڈ ہے جو ک...
2,prompt,پاکستان کا دارالحکومت ایبادان ہے,Python ایک بہت ہی مقبول اور موثر پروگرامنگ زبا...
3,prompt,پاکستان کا دارالحکومت ایبادان ہے,مصنوعی ذہانت یا مصنوعی ذہانت (انگریزی: Artific...
4,prompt,پاکستان کا دارالحکومت ایبادان ہے,SQL اور NoSQL دونوں ڈیٹا بزنس کے لیے ایک اہم ذ...


In [45]:
def judge_response(prompt, response_a, response_b):
  prompt_text = f"""
You are an impartial evaluator.

Compare the following two responses to the same Urdu question.

Question:
{prompt}

Response A:
{response_a}

Response B:
{response_b}

Evaluate based on:
- Correctness
- Helpfulness
- Clarity
- Completeness
- Fluency

Return ONLY valid JSON in this format:

{{
    "winner": "A" or "B" or "Tie",
    "reason": "<short explanation>"
}}
"""

  completion = client.chat.completions.create(
      model='llama-3.3-70b-versatile',
      messages=[
          {
              'role': 'user',
              'content': prompt_text
          }
      ],
      temperature=0
  )

  return completion.choices[0].message.content

In [46]:
judge_results = []

for _, row in tqdm(evaluation_df.iterrows(), total=len(evaluation_df)):
  result = judge_response(
      row['prompt'],
      row['sft_response'],
      row['dpo_response'],
  )

  judge_results.append(result)

evaluation_df['judge_result'] = judge_results

evaluation_df.head()

100%|██████████| 100/100 [03:57<00:00,  2.37s/it]


,prompt,sft_response,dpo_response,judge_result
0,prompt,پاکستان کا دارالحکومت ایبادان ہے,پاکستان کا دارالحکومت اسلام آباد ہے۔,"{\n ""winner"": ""B"",\n ""reason"": ""Response..."
1,prompt,پاکستان کا دارالحکومت ایبادان ہے,مشین لرننگ (Machine Learning) ایک فیلڈ ہے جو ک...,"{\n ""winner"": ""B"",\n ""reason"": ""Response..."
2,prompt,پاکستان کا دارالحکومت ایبادان ہے,Python ایک بہت ہی مقبول اور موثر پروگرامنگ زبا...,"{\n ""winner"": ""B"",\n ""reason"": ""Response..."
3,prompt,پاکستان کا دارالحکومت ایبادان ہے,مصنوعی ذہانت یا مصنوعی ذہانت (انگریزی: Artific...,"{\n ""winner"": ""B"",\n ""reason"": ""Response..."
4,prompt,پاکستان کا دارالحکومت ایبادان ہے,SQL اور NoSQL دونوں ڈیٹا بزنس کے لیے ایک اہم ذ...,"```json\n{\n ""winner"": ""B"",\n ""reason"": ..."


In [47]:
import json

parsed_results = []

for result in evaluation_df['judge_result']:
  try:
    parsed = json.loads(result)
  except Exception:
    parsed = {
        'winner': 'Error',
        'reason': result
    }

  parsed_results.append(parsed)

parsed_df = pd.DataFrame(parsed_results)

evaluation_df = pd.concat([evaluation_df, parsed_df],
                          axis=1
                          )

evaluation_df.head()

,prompt,sft_response,dpo_response,judge_result,winner,reason
0,prompt,پاکستان کا دارالحکومت ایبادان ہے,پاکستان کا دارالحکومت اسلام آباد ہے۔,"{\n ""winner"": ""B"",\n ""reason"": ""Response...",B,Response B provides the correct capital of Pak...
1,prompt,پاکستان کا دارالحکومت ایبادان ہے,مشین لرننگ (Machine Learning) ایک فیلڈ ہے جو ک...,"{\n ""winner"": ""B"",\n ""reason"": ""Response...",B,"Response B is more helpful, clear, and complet..."
2,prompt,پاکستان کا دارالحکومت ایبادان ہے,Python ایک بہت ہی مقبول اور موثر پروگرامنگ زبا...,"{\n ""winner"": ""B"",\n ""reason"": ""Response...",B,"Response B is more helpful, clear, and complet..."
3,prompt,پاکستان کا دارالحکومت ایبادان ہے,مصنوعی ذہانت یا مصنوعی ذہانت (انگریزی: Artific...,"{\n ""winner"": ""B"",\n ""reason"": ""Response...",B,"Response B is more helpful and clear, although..."
4,prompt,پاکستان کا دارالحکومت ایبادان ہے,SQL اور NoSQL دونوں ڈیٹا بزنس کے لیے ایک اہم ذ...,"```json\n{\n ""winner"": ""B"",\n ""reason"": ...",Error,"```json\n{\n ""winner"": ""B"",\n ""reason"": ..."


In [51]:
winner_counts = evaluation_df["winner"].value_counts()

print(winner_counts)

winner
Error    73
A        14
B        13
Name: count, dtype: int64


In [49]:
sft_wins = (evaluation_df["winner"] == "A").sum()
dpo_wins = (evaluation_df["winner"] == "B").sum()
ties = (evaluation_df["winner"] == "Tie").sum()

print(f"SFT Wins : {sft_wins}")
print(f"DPO Wins : {dpo_wins}")
print(f"Ties     : {ties}")

SFT Wins : 14
DPO Wins : 13
Ties     : 0


In [50]:
evaluation_df.to_csv(
    "llm_judge_results.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!
